# MedSentiX: Full Experimental Pipeline
A consolidated, paper-friendly notebook for the complete end-to-end MedSentiX workflow.

## 1. Environment and Reproducibility
This section configures relative project paths, random seed 42, and automatic CUDA/MPS/CPU device detection.

In [ ]:
# DEV MODE keeps notebook runs small during development; set to False for full DGX experiments.
DEV_MODE = True
SAMPLE_SIZE = 10000

# Resolve the project root from either the repository root or notebooks/ directory.
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


### Device Utility Implementation
The utility is included directly so this notebook remains self-contained.

In [ ]:
"""Device and reproducibility utilities for the MedSentiX project.

Every notebook and training module imports from this file so CUDA, Apple MPS,
and CPU execution are selected consistently without hardcoding a backend.
"""


import os
import random

import numpy as np
import torch


RANDOM_SEED = 42


def get_device() -> torch.device:
    """Return the best available torch device and print the selected backend."""
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print(f"Selected device: CUDA ({torch.cuda.get_device_name(0)})")
    elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Selected device: Apple Silicon MPS")
    else:
        device = torch.device("cpu")
        print("Selected device: CPU")
    return device


def set_seed(seed: int = RANDOM_SEED) -> None:
    """Seed Python, NumPy, and PyTorch for reproducible experiments."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


__all__ = ["RANDOM_SEED", "get_device", "set_seed"]


In [ ]:
set_seed(RANDOM_SEED)
DEVICE = get_device()


## 2. Dataset Preparation
The three datasets are loaded, inspected, cleaned, labeled, and split using the exact rules in the specification.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split


# Shared preprocessing utilities used identically across datasets.
import html
import re

LABEL2ID = {"Negative": 0, "Neutral": 1, "Positive": 2}
ID2LABEL = {0: "Negative", 1: "Neutral", 2: "Positive"}
NUM_CLASSES = 3


def clean_text(value):
    # Apply the exact text cleaning pipeline from the specification.
    if not isinstance(value, str):
        return ""
    text = value.lower()
    text = html.unescape(text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s'-]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def label_from_rating_10(rating):
    # Map Drugs.com 1-10 ratings to Negative/Neutral/Positive labels.
    rating = float(rating)
    if 1 <= rating <= 4:
        return 0
    if 5 <= rating <= 6:
        return 1
    if 7 <= rating <= 10:
        return 2
    return None


def label_from_rating_5(rating):
    # Map 1-5 ratings to Negative/Neutral/Positive labels.
    rating = float(rating)
    if 1 <= rating <= 2:
        return 0
    if rating == 3:
        return 1
    if 4 <= rating <= 5:
        return 2
    return None


def label_from_effectiveness(value):
    # Map Druglib effectiveness text to the three-class label space.
    mapping = {
        "Ineffective": 0,
        "Marginally Effective": 1,
        "Moderately Effective": 1,
        "Considerably Effective": 2,
        "Highly Effective": 2,
    }
    return mapping.get(value, None)


def label_from_side_effects(value):
    # Map Druglib side-effect severity text to the three-class label space.
    mapping = {
        "Severe Side Effects": 0,
        "Extremely Severe Side Effects": 0,
        "Mild Side Effects": 1,
        "Moderate Side Effects": 1,
        "No Side Effects": 2,
    }
    return mapping.get(value, None)


### 2.1 Dataset Loading and Inspection

In [ ]:
raw_dir = PROJECT_ROOT / "data/raw"
drugs_train_raw = pd.read_csv(raw_dir / "drugsComTrain_raw.csv")
drugs_test_raw = pd.read_csv(raw_dir / "drugsComTest_raw.csv")
druglib_train_raw = pd.read_csv(raw_dir / "drugLibTrain_raw.tsv", sep="\t")
druglib_test_raw = pd.read_csv(raw_dir / "drugLibTest_raw.tsv", sep="\t")
webmd_raw = pd.read_csv(raw_dir / "webmd_raw.csv")
for name, frame in {
    "drugs_train_raw": drugs_train_raw,
    "drugs_test_raw": drugs_test_raw,
    "druglib_train_raw": druglib_train_raw,
    "druglib_test_raw": druglib_test_raw,
    "webmd_raw": webmd_raw,
}.items():
    print(name, frame.shape)
    print(frame.columns.tolist())


### 2.2 Drugs.com Preprocessing

In [ ]:
drugs_df = pd.concat([drugs_train_raw, drugs_test_raw], ignore_index=True)
if DEV_MODE:
    drugs_df = drugs_df.head(SAMPLE_SIZE)
print("Null counts:")
print(drugs_df.isna().sum())
print("Duplicate rows:", drugs_df.duplicated().sum())
drugs_df = drugs_df[["review", "rating"]].dropna().drop_duplicates().copy()
drugs_df["review"] = drugs_df["review"].apply(clean_text)
drugs_df = drugs_df[drugs_df["review"].str.len() > 0].copy()
drugs_df["label"] = drugs_df["rating"].apply(label_from_rating_10)
drugs_df = drugs_df.dropna(subset=["label"]).copy()
drugs_df["label"] = drugs_df["label"].astype(int)
drugs_df.to_csv(PROJECT_ROOT / "data/processed/drugs_com_clean.csv", index=False)
print(drugs_df.shape)
print(drugs_df["label"].value_counts(normalize=True).sort_index())


### 2.3 Druglib.com Preprocessing

In [ ]:
druglib_df = pd.concat([druglib_train_raw, druglib_test_raw], ignore_index=True)
if DEV_MODE:
    druglib_df = druglib_df.head(SAMPLE_SIZE)
druglib_df = druglib_df[["rating", "effectiveness", "sideEffects", "benefitsReview", "sideEffectsReview", "commentsReview"]].dropna().drop_duplicates().copy()
for column in ["benefitsReview", "sideEffectsReview", "commentsReview"]:
    druglib_df[column] = druglib_df[column].apply(clean_text)
druglib_df["review"] = (
    druglib_df["benefitsReview"].astype(str) + " " +
    druglib_df["sideEffectsReview"].astype(str) + " " +
    druglib_df["commentsReview"].astype(str)
).str.strip()
druglib_df["label"] = druglib_df["rating"].apply(label_from_rating_5)
druglib_df["efficacy_label"] = druglib_df["effectiveness"].apply(label_from_effectiveness)
druglib_df["side_effects_label"] = druglib_df["sideEffects"].apply(label_from_side_effects)
druglib_df["ease_label"] = -100
druglib_df["satisfaction_label"] = -100
druglib_df = druglib_df.dropna(subset=["label", "efficacy_label", "side_effects_label"]).copy()
for column in ["label", "efficacy_label", "side_effects_label", "ease_label", "satisfaction_label"]:
    druglib_df[column] = druglib_df[column].astype(int)
druglib_df.to_csv(PROJECT_ROOT / "data/processed/druglib_clean.csv", index=False)
print(druglib_df.shape)


### 2.4 WebMD Preprocessing

In [ ]:
webmd_df = webmd_raw.rename(columns={
    "Reviews": "review",
    "Effectiveness": "effectiveness",
    "EaseofUse": "easeOfUse",
    "Satisfaction": "satisfaction",
    "Sides": "sideEffectsReview",
})
if "rating" not in webmd_df.columns:
    webmd_df["rating"] = webmd_df["satisfaction"]
if DEV_MODE:
    webmd_df = webmd_df.head(SAMPLE_SIZE)
webmd_df = webmd_df[["review", "rating", "effectiveness", "easeOfUse", "satisfaction", "sideEffectsReview"]].dropna().drop_duplicates().copy()
webmd_df["review"] = webmd_df["review"].apply(clean_text)
webmd_df["sideEffectsReview"] = webmd_df["sideEffectsReview"].apply(clean_text)
webmd_df = webmd_df[(webmd_df["review"].str.len() > 0) & (webmd_df["sideEffectsReview"].str.len() > 0)].copy()
webmd_df["label"] = webmd_df["rating"].apply(label_from_rating_5)
webmd_df["efficacy_label"] = webmd_df["effectiveness"].apply(label_from_rating_5)
webmd_df["ease_label"] = webmd_df["easeOfUse"].apply(label_from_rating_5)
webmd_df["satisfaction_label"] = webmd_df["satisfaction"].apply(label_from_rating_5)
webmd_df["side_effects_label"] = -100
webmd_df = webmd_df.dropna(subset=["label", "efficacy_label", "ease_label", "satisfaction_label"]).copy()
for column in ["label", "efficacy_label", "ease_label", "satisfaction_label", "side_effects_label"]:
    webmd_df[column] = webmd_df[column].astype(int)
webmd_df.to_csv(PROJECT_ROOT / "data/processed/webmd_clean.csv", index=False)
print(webmd_df.shape)


### 2.5 Dataset Splitting

In [ ]:
def stratified_80_10_10(frame):
    train_df, temp_df = train_test_split(frame, test_size=0.20, random_state=RANDOM_SEED, stratify=frame["label"])
    val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=RANDOM_SEED, stratify=temp_df["label"])
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)


split_dir = PROJECT_ROOT / "data/splits"
split_dir.mkdir(parents=True, exist_ok=True)
for name, frame in {"drugs_com": drugs_df, "webmd": webmd_df, "druglib": druglib_df}.items():
    for split_name, split_df in zip(["train", "val", "test"], stratified_80_10_10(frame)):
        split_df.to_csv(split_dir / f"{name}_{split_name}.csv", index=False)
        print(name, split_name, split_df.shape, split_df["label"].value_counts(normalize=True).sort_index().to_dict())


## 3. Baseline Models
The following cell contains the actual baseline model implementations used by the modular notebooks.

In [ ]:
r'''
"""Baseline models and training utilities for MedSentiX.

The code in this module follows the paper specification for all seven
baselines. Notebook cells call these helpers so the implementation remains
reproducible while keeping each notebook readable.
"""


import json
import math
import re
import time
from collections import Counter
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
)
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

try:
    from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
except Exception:  # pragma: no cover - notebooks surface dependency errors clearly.
    AutoModel = None
    AutoTokenizer = None
    get_linear_schedule_with_warmup = None

# Device utilities are defined in this notebook.


# Project-level constants mirror the paper specification.
# PROJECT_ROOT is supplied by the notebook environment
LABEL2ID = {"Negative": 0, "Neutral": 1, "Positive": 2}
ID2LABEL = {0: "Negative", 1: "Neutral", 2: "Positive"}
NUM_CLASSES = 3
GLOVE_DIM = 100
GLOVE_MAX_LEN = 256
TRANSFORMER_MAX_LEN = 512
BATCH_SIZE = 16
EPOCHS = 10
DROPOUT = 0.3
CLASS_WEIGHTS = torch.tensor([1.5, 2.0, 1.0], dtype=torch.float)


def ensure_output_dirs(project_root: Path = PROJECT_ROOT) -> None:
    """Create checkpoint and result directories used by the baseline workflow."""
    for rel in [
        "checkpoints/baselines",
        "results/tables",
        "results/figures/confusion_matrices",
        "results/figures/training_curves",
    ]:
        (project_root / rel).mkdir(parents=True, exist_ok=True)


def parameter_count(model: nn.Module) -> int:
    """Count trainable and frozen parameters for reporting in result tables."""
    return int(sum(param.numel() for param in model.parameters()))


def compute_classification_metrics(
    y_true: Sequence[int],
    y_pred: Sequence[int],
    inference_ms_per_sample: float = math.nan,
    params: float = math.nan,
) -> Dict[str, float]:
    """Compute every classification metric required by the paper specification."""
    labels = [0, 1, 2]
    per_class_f1 = f1_score(y_true, y_pred, labels=labels, average=None, zero_division=0)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "negative_f1": per_class_f1[0],
        "neutral_f1": per_class_f1[1],
        "positive_f1": per_class_f1[2],
        "mcc": matthews_corrcoef(y_true, y_pred),
        "cohen_kappa": cohen_kappa_score(y_true, y_pred),
        "inference_ms_per_sample": inference_ms_per_sample,
        "parameter_count": params,
    }


def save_confusion_matrix(
    y_true: Sequence[int],
    y_pred: Sequence[int],
    model_name: str,
    project_root: Path = PROJECT_ROOT,
) -> np.ndarray:
    """Save a normalized confusion matrix figure and return the matrix values."""
    output_dir = project_root / "results/figures/confusion_matrices"
    output_dir.mkdir(parents=True, exist_ok=True)
    matrix = confusion_matrix(y_true, y_pred, labels=[0, 1, 2], normalize="true")
    plt.figure(figsize=(6, 5))
    sns.heatmap(
        matrix,
        annot=True,
        fmt=".2f",
        cmap="Blues",
        xticklabels=[ID2LABEL[i] for i in range(NUM_CLASSES)],
        yticklabels=[ID2LABEL[i] for i in range(NUM_CLASSES)],
    )
    plt.title(f"{model_name} Normalized Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(output_dir / f"{model_name}_confusion_matrix.png", dpi=300)
    plt.close()
    return matrix


def save_training_curve(history: Dict[str, List[float]], model_name: str, project_root: Path = PROJECT_ROOT) -> None:
    """Save loss and validation accuracy curves for neural baselines."""
    if not history:
        return
    output_dir = project_root / "results/figures/training_curves"
    output_dir.mkdir(parents=True, exist_ok=True)
    plt.figure(figsize=(7, 4))
    if history.get("train_loss"):
        plt.plot(history["train_loss"], label="Train Loss")
    if history.get("val_accuracy"):
        plt.plot(history["val_accuracy"], label="Validation Accuracy")
    plt.title(f"{model_name} Training Curve")
    plt.xlabel("Epoch")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_dir / f"{model_name}_training_curve.png", dpi=300)
    plt.close()


def simple_tokenize(text: str) -> List[str]:
    """Tokenize text for GloVe baselines while preserving negations and numbers."""
    return re.findall(r"[a-z0-9'-]+", str(text).lower())


def build_vocab(texts: Iterable[str], max_vocab: int = 50000) -> Dict[str, int]:
    """Build a capped vocabulary with PAD=0 and UNK=1 special tokens."""
    counter: Counter[str] = Counter()
    for text in texts:
        counter.update(simple_tokenize(text))
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for token, _ in counter.most_common(max_vocab - len(vocab)):
        vocab[token] = len(vocab)
    return vocab


def load_glove_embeddings(
    glove_path: Path,
    vocab: Dict[str, int],
    embedding_dim: int = GLOVE_DIM,
    seed: int = RANDOM_SEED,
) -> np.ndarray:
    """Load only the GloVe vectors needed for the current vocabulary."""
    rng = np.random.default_rng(seed)
    matrix = rng.normal(0.0, 0.05, size=(len(vocab), embedding_dim)).astype(np.float32)
    matrix[vocab["<PAD>"]] = np.zeros(embedding_dim, dtype=np.float32)

    with glove_path.open("r", encoding="utf-8", errors="ignore") as handle:
        for line in handle:
            pieces = line.rstrip().split(" ")
            token = pieces[0]
            if token in vocab and len(pieces) == embedding_dim + 1:
                matrix[vocab[token]] = np.asarray(pieces[1:], dtype=np.float32)
    return matrix


class GloveReviewDataset(Dataset):
    """Numericalize review text for BiLSTM, BiLSTM-CNN, and Double-BiGRU baselines."""

    def __init__(self, texts: Sequence[str], labels: Sequence[int], vocab: Dict[str, int], max_len: int = GLOVE_MAX_LEN):
        self.texts = list(texts)
        self.labels = list(labels)
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, index: int) -> Dict[str, torch.Tensor]:
        tokens = simple_tokenize(self.texts[index])[: self.max_len]
        ids = [self.vocab.get(token, self.vocab["<UNK>"]) for token in tokens]
        mask = [1] * len(ids)
        padding = self.max_len - len(ids)
        ids.extend([self.vocab["<PAD>"]] * padding)
        mask.extend([0] * padding)
        return {
            "input_ids": torch.tensor(ids, dtype=torch.long),
            "attention_mask": torch.tensor(mask, dtype=torch.float),
            "labels": torch.tensor(self.labels[index], dtype=torch.long),
        }


class TransformerReviewDataset(Dataset):
    """Tokenize review text for BERT, RoBERTa, and BioBERT baselines."""

    def __init__(self, texts: Sequence[str], labels: Sequence[int], tokenizer, max_len: int = TRANSFORMER_MAX_LEN):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, index: int) -> Dict[str, torch.Tensor]:
        encoded = self.tokenizer(
            str(self.texts[index]),
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[index], dtype=torch.long),
        }


def masked_mean(sequence: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """Mean-pool sequence states over non-padding tokens."""
    mask = mask.unsqueeze(-1).float()
    return (sequence * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)


class DoubleBiGRUClassifier(nn.Module):
    """Two stacked BiGRU layers with GloVe embeddings."""

    def __init__(self, embedding_matrix: np.ndarray, dropout: float = DROPOUT):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(torch.tensor(embedding_matrix), freeze=False, padding_idx=0)
        self.gru1 = nn.GRU(GLOVE_DIM, 256, batch_first=True, bidirectional=True)
        self.gru2 = nn.GRU(512, 256, batch_first=True, bidirectional=True)
        self.classifier = nn.Sequential(nn.Linear(512, 128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128, NUM_CLASSES))

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        embedded = self.embedding(input_ids)
        out, _ = self.gru1(embedded)
        out, _ = self.gru2(out)
        return self.classifier(masked_mean(out, attention_mask))


class BiLSTMGloveClassifier(nn.Module):
    """Two-layer bidirectional LSTM with fine-tuned GloVe embeddings."""

    def __init__(self, embedding_matrix: np.ndarray, dropout: float = DROPOUT):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(torch.tensor(embedding_matrix), freeze=False, padding_idx=0)
        self.lstm = nn.LSTM(GLOVE_DIM, 256, num_layers=2, batch_first=True, bidirectional=True, dropout=dropout)
        self.classifier = nn.Sequential(nn.Linear(512, 128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128, NUM_CLASSES))

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        out, _ = self.lstm(self.embedding(input_ids))
        return self.classifier(masked_mean(out, attention_mask))


class BiLSTMCNNGloveClassifier(nn.Module):
    """CNN feature extractor followed by a BiLSTM classifier."""

    def __init__(self, embedding_matrix: np.ndarray, dropout: float = DROPOUT):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(torch.tensor(embedding_matrix), freeze=False, padding_idx=0)
        self.conv = nn.Conv1d(GLOVE_DIM, 64, kernel_size=5, padding=2)
        self.pool = nn.MaxPool1d(4)
        self.lstm = nn.LSTM(64, 64, batch_first=True, bidirectional=True, dropout=dropout)
        self.classifier = nn.Linear(128, NUM_CLASSES)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        embedded = self.embedding(input_ids).transpose(1, 2)
        convolved = torch.relu(self.conv(embedded))
        pooled = self.pool(convolved).transpose(1, 2)
        pooled_mask = attention_mask[:, ::4][:, : pooled.size(1)]
        if pooled_mask.size(1) < pooled.size(1):
            pad = pooled.size(1) - pooled_mask.size(1)
            pooled_mask = torch.nn.functional.pad(pooled_mask, (0, pad))
        out, _ = self.lstm(pooled)
        return self.classifier(masked_mean(out, pooled_mask))


class TransformerBiLSTMClassifier(nn.Module):
    """BERT/RoBERTa encoder with a two-layer BiLSTM classification head."""

    def __init__(self, model_name: str, dropout: float = DROPOUT):
        super().__init__()
        if AutoModel is None:
            raise ImportError("transformers is required for transformer baselines.")
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.lstm = nn.LSTM(hidden_size, 256, num_layers=2, batch_first=True, bidirectional=True, dropout=dropout)
        self.norm = nn.LayerNorm(512)
        self.classifier = nn.Sequential(nn.Linear(512, 128), nn.Dropout(dropout), nn.Linear(128, NUM_CLASSES))

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        hidden = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        out, _ = self.lstm(hidden)
        out = self.norm(out)
        return self.classifier(masked_mean(out, attention_mask))


class BioBERTStandaloneClassifier(nn.Module):
    """BioBERT baseline using CLS pooling without BiLSTM or guided attention."""

    def __init__(self, model_name: str = "dmis-lab/biobert-base-cased-v1.2", dropout: float = DROPOUT):
        super().__init__()
        if AutoModel is None:
            raise ImportError("transformers is required for BioBERT baseline.")
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.classifier = nn.Sequential(nn.Linear(hidden_size, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, NUM_CLASSES))

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        cls_state = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0, :]
        return self.classifier(cls_state)


def train_svm_baseline(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    checkpoint_path: Path,
    model_name: str = "svm",
    project_root: Path = PROJECT_ROOT,
) -> Dict[str, float]:
    """Train and evaluate the TF-IDF + LinearSVC baseline on Drugs.com."""
    ensure_output_dirs(project_root)
    pipeline = Pipeline(
        [
            ("tfidf", TfidfVectorizer(max_features=50000, ngram_range=(1, 2), sublinear_tf=True)),
            ("svm", LinearSVC(C=1.0, max_iter=2000, random_state=RANDOM_SEED)),
        ]
    )
    pipeline.fit(train_df["review"].astype(str), train_df["label"].astype(int))

    start = time.perf_counter()
    predictions = pipeline.predict(test_df["review"].astype(str))
    elapsed = time.perf_counter() - start
    inference_ms = 1000.0 * elapsed / max(len(test_df), 1)

    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(pipeline, checkpoint_path)
    metrics = compute_classification_metrics(test_df["label"].astype(int), predictions, inference_ms, params=0)
    save_confusion_matrix(test_df["label"].astype(int), predictions, model_name, project_root)
    return {"model": model_name, **metrics}


def evaluate_neural_model(model: nn.Module, loader: DataLoader, device: torch.device) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    """Evaluate a neural baseline and return metrics, labels, and predictions."""
    model.eval()
    labels: List[int] = []
    predictions: List[int] = []
    start = time.perf_counter()
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            predictions.extend(torch.argmax(logits, dim=1).detach().cpu().tolist())
            labels.extend(batch["labels"].detach().cpu().tolist())
    elapsed = time.perf_counter() - start
    inference_ms = 1000.0 * elapsed / max(len(labels), 1)
    metrics = compute_classification_metrics(labels, predictions, inference_ms, parameter_count(model))
    return metrics, np.asarray(labels), np.asarray(predictions)


def train_neural_classifier(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    test_loader: DataLoader,
    checkpoint_path: Path,
    model_name: str,
    lr: float,
    use_scheduler: bool = False,
    project_root: Path = PROJECT_ROOT,
    epochs: int = EPOCHS,
) -> Dict[str, float]:
    """Train a neural baseline with class-weighted cross entropy and early checkpointing."""
    set_seed(RANDOM_SEED)
    ensure_output_dirs(project_root)
    device = get_device()
    model = model.to(device)
    criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS.to(device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr) if use_scheduler else torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = None
    if use_scheduler and get_linear_schedule_with_warmup is not None:
        total_steps = len(train_loader) * epochs
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=int(0.1 * total_steps),
            num_training_steps=total_steps,
        )

    best_val_accuracy = -1.0
    history = {"train_loss": [], "val_accuracy": []}
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

    for epoch in range(epochs):
        model.train()
        losses: List[float] = []
        for batch in tqdm(train_loader, desc=f"{model_name} epoch {epoch + 1}/{epochs}"):
            optimizer.zero_grad(set_to_none=True)
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            loss = criterion(model(input_ids=input_ids, attention_mask=attention_mask), labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            if scheduler is not None:
                scheduler.step()
            losses.append(float(loss.detach().cpu()))

        val_metrics, _, _ = evaluate_neural_model(model, val_loader, device)
        history["train_loss"].append(float(np.mean(losses)))
        history["val_accuracy"].append(float(val_metrics["accuracy"]))
        if val_metrics["accuracy"] > best_val_accuracy:
            best_val_accuracy = val_metrics["accuracy"]
            torch.save(model.state_dict(), checkpoint_path)

    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    test_metrics, y_true, y_pred = evaluate_neural_model(model, test_loader, device)
    save_confusion_matrix(y_true, y_pred, model_name, project_root)
    save_training_curve(history, model_name, project_root)
    with (checkpoint_path.with_suffix(".history.json")).open("w", encoding="utf-8") as handle:
        json.dump(history, handle, indent=2)
    return {"model": model_name, **test_metrics}


def make_glove_loaders(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    batch_size: int = BATCH_SIZE,
    project_root: Path = PROJECT_ROOT,
) -> Tuple[DataLoader, DataLoader, DataLoader, np.ndarray]:
    """Build vocabulary, embedding matrix, and dataloaders for GloVe baselines."""
    vocab = build_vocab(train_df["review"].astype(str))
    embeddings = load_glove_embeddings(project_root / "glove/glove.6B.100d.txt", vocab)
    train_ds = GloveReviewDataset(train_df["review"], train_df["label"].astype(int), vocab)
    val_ds = GloveReviewDataset(val_df["review"], val_df["label"].astype(int), vocab)
    test_ds = GloveReviewDataset(test_df["review"], test_df["label"].astype(int), vocab)
    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True),
        DataLoader(val_ds, batch_size=batch_size),
        DataLoader(test_ds, batch_size=batch_size),
        embeddings,
    )


def make_transformer_loaders(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    model_name: str,
    batch_size: int = BATCH_SIZE,
) -> Tuple[DataLoader, DataLoader, DataLoader]:
    """Create tokenizer-backed dataloaders for transformer baselines."""
    if AutoTokenizer is None:
        raise ImportError("transformers is required for transformer baselines.")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    train_ds = TransformerReviewDataset(train_df["review"], train_df["label"].astype(int), tokenizer)
    val_ds = TransformerReviewDataset(val_df["review"], val_df["label"].astype(int), tokenizer)
    test_ds = TransformerReviewDataset(test_df["review"], test_df["label"].astype(int), tokenizer)
    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True),
        DataLoader(val_ds, batch_size=batch_size),
        DataLoader(test_ds, batch_size=batch_size),
    )


def train_all_baselines(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    project_root: Path = PROJECT_ROOT,
    epochs: int = EPOCHS,
) -> pd.DataFrame:
    """Train all seven baselines and save the consolidated baseline result table."""
    ensure_output_dirs(project_root)
    results: List[Dict[str, float]] = []
    checkpoints = project_root / "checkpoints/baselines"

    results.append(train_svm_baseline(train_df, test_df, checkpoints / "svm.pkl", project_root=project_root))

    glove_train, glove_val, glove_test, embeddings = make_glove_loaders(train_df, val_df, test_df, project_root=project_root)
    glove_jobs = [
        ("double_bigru", DoubleBiGRUClassifier(embeddings), checkpoints / "double_bigru.pt"),
        ("bilstm", BiLSTMGloveClassifier(embeddings), checkpoints / "bilstm.pt"),
        ("bilstm_cnn", BiLSTMCNNGloveClassifier(embeddings), checkpoints / "bilstm_cnn.pt"),
    ]
    for name, model, path in glove_jobs:
        results.append(train_neural_classifier(model, glove_train, glove_val, glove_test, path, name, lr=1e-3, epochs=epochs))

    transformer_jobs = [
        ("bert_bilstm", "bert-base-uncased", TransformerBiLSTMClassifier("bert-base-uncased"), checkpoints / "bert_bilstm.pt"),
        ("roberta_bilstm", "roberta-base", TransformerBiLSTMClassifier("roberta-base"), checkpoints / "roberta_bilstm.pt"),
        ("biobert_standalone", "dmis-lab/biobert-base-cased-v1.2", BioBERTStandaloneClassifier(), checkpoints / "biobert_standalone.pt"),
    ]
    for name, tokenizer_name, model, path in transformer_jobs:
        train_loader, val_loader, test_loader = make_transformer_loaders(train_df, val_df, test_df, tokenizer_name)
        results.append(train_neural_classifier(model, train_loader, val_loader, test_loader, path, name, lr=2e-5, use_scheduler=True, epochs=epochs))

    result_df = pd.DataFrame(results)
    result_df.to_csv(project_root / "results/tables/baseline_results.csv", index=False)
    return result_df
'''
# The maintained baseline implementation lives in models/baselines.py.
from models.baselines import train_all_baselines


### 3.1 Train Baselines

In [ ]:
train_df = pd.read_csv(PROJECT_ROOT / "data/splits/drugs_com_train.csv")
val_df = pd.read_csv(PROJECT_ROOT / "data/splits/drugs_com_val.csv")
test_df = pd.read_csv(PROJECT_ROOT / "data/splits/drugs_com_test.csv")
if DEV_MODE:
    train_df = train_df.head(SAMPLE_SIZE)
    val_df = val_df.head(max(100, SAMPLE_SIZE // 5))
    test_df = test_df.head(max(100, SAMPLE_SIZE // 5))
baseline_results = train_all_baselines(train_df, val_df, test_df, project_root=PROJECT_ROOT, epochs=10)
display(baseline_results)
from utils.memory import cleanup_memory
del train_df, val_df, test_df
cleanup_memory()


## 4. MedSentiX Architecture
The actual MedSentiX implementation is included below: BioBERT, BiLSTM, guided attention, classification head, losses, and experiment utilities.

In [ ]:
r'''
"""MedSentiX architecture, training, evaluation, ablation, and explainability.

This module implements the four-component model described in the specification:
BioBERT, BiLSTM, guided multi-head aspect attention, and the classification
head. It also contains reusable experiment helpers used by the notebooks.
"""


import json
import math
import time
from pathlib import Path
from typing import Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import accuracy_score, f1_score
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

try:
    import shap
except Exception:  # pragma: no cover - SHAP notebooks report this directly.
    shap = None

try:
    from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
except Exception:  # pragma: no cover - notebooks surface dependency errors clearly.
    AutoModel = None
    AutoTokenizer = None
    get_linear_schedule_with_warmup = None

# Baseline utilities are defined above in this notebook.
# Device utilities are defined in this notebook.


# Training constants are kept identical to the specification.
# PROJECT_ROOT is supplied by the notebook environment
BIOBERT_MODEL_NAME = "dmis-lab/biobert-base-cased-v1.2"
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
EPOCHS = 10
WARMUP_RATIO = 0.1
DROPOUT = 0.3
MAX_LEN = 512
LAMBDA_ASPECT = 0.3
GRADIENT_CLIP = 1.0
EARLY_STOPPING_PATIENCE = 3
MISSING_ASPECT_LABEL = -100
ASPECT_NAMES = ["efficacy", "side_effects", "ease_of_use", "overall_satisfaction"]
ASPECT_COLUMNS = ["efficacy_label", "side_effects_label", "ease_label", "satisfaction_label"]


def ensure_medsentix_dirs(project_root: Path = PROJECT_ROOT) -> None:
    """Create all output directories used by MedSentiX experiments."""
    for rel in [
        "checkpoints/medsentix",
        "results/tables",
        "results/figures/confusion_matrices",
        "results/figures/training_curves",
        "results/figures/attention_heatmaps",
        "results/figures/shap_plots",
    ]:
        (project_root / rel).mkdir(parents=True, exist_ok=True)


def masked_mean(sequence: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """Mean-pool a sequence over valid, non-padding tokens."""
    mask = mask.unsqueeze(-1).float()
    return (sequence * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)


class BioBERTEncoder(nn.Module):
    """Fine-tuned BioBERT encoder with dropout applied to token states."""

    def __init__(self, model_name: str = BIOBERT_MODEL_NAME, dropout: float = DROPOUT):
        super().__init__()
        if AutoModel is None:
            raise ImportError("transformers is required to instantiate BioBERTEncoder.")
        self.biobert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        outputs = self.biobert(input_ids=input_ids, attention_mask=attention_mask)
        return self.dropout(outputs.last_hidden_state)


class BiLSTMLayer(nn.Module):
    """Two-layer bidirectional LSTM that maps BioBERT states to 512 dimensions."""

    def __init__(self, input_size: int = 768, hidden_size: int = 256, num_layers: int = 2, dropout: float = DROPOUT):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout,
        )
        self.layer_norm = nn.LayerNorm(hidden_size * 2)

    def forward(self, sequence: torch.Tensor) -> torch.Tensor:
        output, _ = self.lstm(sequence)
        return self.layer_norm(output)


class GuidedMultiHeadAspectAttention(nn.Module):
    """Four-head attention layer with explicit aspect head assignments."""

    def __init__(self, embed_dim: int = 512, num_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.attention = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.layer_norm = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)
        self.attention_weights: Optional[torch.Tensor] = None

    def forward(self, sequence: torch.Tensor, attention_mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, List[torch.Tensor]]:
        key_padding_mask = attention_mask.eq(0)
        attended, attn_weights = self.attention(
            sequence,
            sequence,
            sequence,
            key_padding_mask=key_padding_mask,
            need_weights=True,
            average_attn_weights=False,
        )
        attended = self.layer_norm(sequence + self.dropout(attended))
        self.attention_weights = attn_weights.detach()
        pooled = masked_mean(attended, attention_mask)

        # Split the attended representation into four head-specific projections
        # for auxiliary aspect supervision.
        head_sequences = torch.chunk(attended, self.num_heads, dim=-1)
        head_outputs = [masked_mean(head_sequence, attention_mask) for head_sequence in head_sequences]
        return pooled, attn_weights, head_outputs


class ClassificationHead(nn.Module):
    """Main sentiment classification head for the pooled MedSentiX state."""

    def __init__(self, input_dim: int = 512, dropout: float = DROPOUT):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, NUM_CLASSES),
        )

    def forward(self, pooled: torch.Tensor) -> torch.Tensor:
        return self.classifier(pooled)


class MedSentiX(nn.Module):
    """Hybrid BioBERT-BiLSTM model with guided aspect-aware attention."""

    def __init__(
        self,
        model_name: str = BIOBERT_MODEL_NAME,
        dropout: float = DROPOUT,
        use_bilstm: bool = True,
        use_attention: bool = True,
        use_auxiliary_heads: bool = True,
    ):
        super().__init__()
        self.use_bilstm = use_bilstm
        self.use_attention = use_attention
        self.use_auxiliary_heads = use_auxiliary_heads
        self.encoder = BioBERTEncoder(model_name=model_name, dropout=dropout)
        encoder_dim = self.encoder.biobert.config.hidden_size
        self.bilstm = BiLSTMLayer(input_size=encoder_dim, dropout=dropout) if use_bilstm else nn.Identity()
        feature_dim = 512 if use_bilstm else encoder_dim
        self.attention = GuidedMultiHeadAspectAttention(embed_dim=feature_dim, num_heads=4) if use_attention else None
        self.classifier = ClassificationHead(input_dim=feature_dim, dropout=dropout)
        head_dim = feature_dim // 4
        self.auxiliary_classifiers = nn.ModuleList([nn.Linear(head_dim, NUM_CLASSES) for _ in range(4)])

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor], List[torch.Tensor]]:
        sequence = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        sequence = self.bilstm(sequence)
        if self.use_attention and self.attention is not None:
            pooled, attn_weights, head_outputs = self.attention(sequence, attention_mask)
        else:
            pooled = masked_mean(sequence, attention_mask)
            attn_weights = None
            head_outputs = [chunk for chunk in torch.chunk(pooled, 4, dim=-1)]
        logits = self.classifier(pooled)
        auxiliary_logits = []
        if self.use_auxiliary_heads:
            auxiliary_logits = [classifier(head_output) for classifier, head_output in zip(self.auxiliary_classifiers, head_outputs)]
        return logits, attn_weights, auxiliary_logits


class MedSentiXDataset(Dataset):
    """Tokenizer-backed dataset with optional labels for four aspect heads."""

    def __init__(self, frame: pd.DataFrame, tokenizer, max_len: int = MAX_LEN):
        self.frame = frame.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int) -> Dict[str, torch.Tensor]:
        row = self.frame.iloc[index]
        encoded = self.tokenizer(
            str(row["review"]),
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        item = {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "labels": torch.tensor(int(row["label"]), dtype=torch.long),
        }
        for column in ASPECT_COLUMNS:
            item[column] = torch.tensor(int(row.get(column, MISSING_ASPECT_LABEL)), dtype=torch.long)
        return item


def _coerce_aspect_column(frame: pd.DataFrame, column: str) -> pd.Series:
    """Return an integer aspect column or a missing-label placeholder."""
    if column not in frame.columns:
        return pd.Series([MISSING_ASPECT_LABEL] * len(frame), index=frame.index, dtype="int64")
    return pd.to_numeric(frame[column], errors="coerce").fillna(MISSING_ASPECT_LABEL).astype("int64")


def standardize_frame(frame: pd.DataFrame, source: str) -> pd.DataFrame:
    """Align each processed split to the shared MedSentiX training schema."""
    output = pd.DataFrame(index=frame.index)
    output["review"] = frame["review"].astype(str)
    output["label"] = pd.to_numeric(frame["label"], errors="coerce").astype("int64")
    for column in ASPECT_COLUMNS:
        output[column] = _coerce_aspect_column(frame, column)
    output["source"] = source
    return output.dropna(subset=["review", "label"]).reset_index(drop=True)


def load_variant_splits(variant: str, project_root: Path = PROJECT_ROOT, dev_mode: bool = False, sample_size: int = 1000) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Load and combine dataset splits for one MedSentiX variant."""
    split_dir = project_root / "data/splits"
    datasets_by_variant = {
        "D": ["drugs_com"],
        "DW": ["drugs_com", "webmd"],
        "DDL": ["drugs_com", "druglib"],
        "Full": ["drugs_com", "webmd", "druglib"],
    }
    if variant not in datasets_by_variant:
        raise ValueError(f"Unknown variant: {variant}")

    combined = {}
    for split in ["train", "val", "test"]:
        pieces = []
        for dataset_name in datasets_by_variant[variant]:
            path = split_dir / f"{dataset_name}_{split}.csv"
            frame = pd.read_csv(path)
            if dev_mode:
                frame = frame.head(sample_size)
            pieces.append(standardize_frame(frame, dataset_name))
        combined[split] = pd.concat(pieces, ignore_index=True).sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)
    return combined["train"], combined["val"], combined["test"]


def make_medsentix_loaders(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    batch_size: int = BATCH_SIZE,
) -> Tuple[DataLoader, DataLoader, DataLoader, object]:
    """Create BioBERT tokenizer and dataloaders for MedSentiX."""
    if AutoTokenizer is None:
        raise ImportError("transformers is required for MedSentiX dataloaders.")
    tokenizer = AutoTokenizer.from_pretrained(BIOBERT_MODEL_NAME)
    train_loader = DataLoader(MedSentiXDataset(train_df, tokenizer), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(MedSentiXDataset(val_df, tokenizer), batch_size=batch_size)
    test_loader = DataLoader(MedSentiXDataset(test_df, tokenizer), batch_size=batch_size)
    return train_loader, val_loader, test_loader, tokenizer


def compute_medsentix_loss(
    logits: torch.Tensor,
    labels: torch.Tensor,
    auxiliary_logits: Sequence[torch.Tensor],
    batch: Mapping[str, torch.Tensor],
    criterion: nn.Module,
    lambda_aspect: float = LAMBDA_ASPECT,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    """Combine main sentiment loss with valid per-head auxiliary aspect losses."""
    main_loss = criterion(logits, labels)
    aux_total = torch.tensor(0.0, device=logits.device)
    aux_values: Dict[str, float] = {}
    for index, (name, column) in enumerate(zip(ASPECT_NAMES, ASPECT_COLUMNS)):
        if index >= len(auxiliary_logits):
            continue
        aspect_labels = batch[column].to(logits.device)
        valid_mask = aspect_labels.ne(MISSING_ASPECT_LABEL)
        if valid_mask.any():
            aux_loss = criterion(auxiliary_logits[index][valid_mask], aspect_labels[valid_mask])
            aux_total = aux_total + aux_loss
            aux_values[f"{name}_loss"] = float(aux_loss.detach().cpu())
    total_loss = main_loss + (lambda_aspect * aux_total if aux_values else 0.0)
    return total_loss, {"main_loss": float(main_loss.detach().cpu()), **aux_values}


def evaluate_medsentix(model: MedSentiX, loader: DataLoader, device: torch.device) -> Tuple[Dict[str, float], np.ndarray, np.ndarray, np.ndarray]:
    """Evaluate a MedSentiX model and return metrics, labels, predictions, probabilities."""
    model.eval()
    labels: List[int] = []
    predictions: List[int] = []
    probabilities: List[np.ndarray] = []
    start = time.perf_counter()
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            logits, _, _ = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.softmax(logits, dim=-1)
            probabilities.extend(probs.detach().cpu().numpy())
            predictions.extend(torch.argmax(probs, dim=1).detach().cpu().tolist())
            labels.extend(batch["labels"].detach().cpu().tolist())
    elapsed = time.perf_counter() - start
    inference_ms = 1000.0 * elapsed / max(len(labels), 1)
    metrics = compute_classification_metrics(labels, predictions, inference_ms, parameter_count(model))
    metrics["ece"] = expected_calibration_error(np.asarray(labels), np.asarray(probabilities))
    return metrics, np.asarray(labels), np.asarray(predictions), np.asarray(probabilities)


def expected_calibration_error(y_true: np.ndarray, probabilities: np.ndarray, n_bins: int = 15) -> float:
    """Compute Expected Calibration Error from predicted probabilities."""
    if len(y_true) == 0:
        return math.nan
    confidences = probabilities.max(axis=1)
    predictions = probabilities.argmax(axis=1)
    accuracies = predictions == y_true
    ece = 0.0
    for lower, upper in zip(np.linspace(0, 1, n_bins, endpoint=False), np.linspace(1 / n_bins, 1, n_bins)):
        mask = (confidences > lower) & (confidences <= upper)
        if mask.any():
            ece += mask.mean() * abs(accuracies[mask].mean() - confidences[mask].mean())
    return float(ece)


def train_medsentix_variant(
    variant: str,
    project_root: Path = PROJECT_ROOT,
    dev_mode: bool = False,
    sample_size: int = 1000,
    epochs: int = EPOCHS,
    lambda_aspect: Optional[float] = None,
    use_bilstm: bool = True,
    use_attention: bool = True,
    use_auxiliary_heads: bool = True,
    checkpoint_name: Optional[str] = None,
    save_variant_results: bool = True,
) -> Tuple[MedSentiX, pd.DataFrame, Dict[str, List[float]]]:
    """Train one MedSentiX variant and save its checkpoint and result row."""
    set_seed(RANDOM_SEED)
    ensure_medsentix_dirs(project_root)
    effective_lambda = 0.0 if variant == "D" else (LAMBDA_ASPECT if lambda_aspect is None else lambda_aspect)
    train_df, val_df, test_df = load_variant_splits(variant, project_root, dev_mode, sample_size)
    train_loader, val_loader, test_loader, _ = make_medsentix_loaders(train_df, val_df, test_df)

    device = get_device()
    model = MedSentiX(use_bilstm=use_bilstm, use_attention=use_attention, use_auxiliary_heads=use_auxiliary_heads).to(device)
    criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS.to(device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(WARMUP_RATIO * total_steps),
        num_training_steps=total_steps,
    )
    checkpoint_stem = checkpoint_name or f"medsentix_{variant}"
    checkpoint_path = project_root / f"checkpoints/medsentix/{checkpoint_stem}.pt"
    history = {"train_loss": [], "val_accuracy": []}
    best_val_accuracy = -1.0
    epochs_without_improvement = 0

    for epoch in range(epochs):
        model.train()
        losses: List[float] = []
        for batch in tqdm(train_loader, desc=f"MedSentiX-{variant} epoch {epoch + 1}/{epochs}"):
            optimizer.zero_grad(set_to_none=True)
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            logits, _, auxiliary_logits = model(input_ids=input_ids, attention_mask=attention_mask)
            loss, _ = compute_medsentix_loss(logits, labels, auxiliary_logits, batch, criterion, effective_lambda)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
            optimizer.step()
            scheduler.step()
            losses.append(float(loss.detach().cpu()))

        val_metrics, _, _, _ = evaluate_medsentix(model, val_loader, device)
        history["train_loss"].append(float(np.mean(losses)))
        history["val_accuracy"].append(float(val_metrics["accuracy"]))
        if val_metrics["accuracy"] > best_val_accuracy:
            best_val_accuracy = val_metrics["accuracy"]
            epochs_without_improvement = 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
                break

    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    metrics, y_true, y_pred, _ = evaluate_medsentix(model, test_loader, device)
    metrics["model"] = f"medsentix_{variant}"
    metrics["cross_dataset_generalization"] = cross_dataset_generalization(model, variant, project_root, device, dev_mode, sample_size)
    save_confusion_matrix(y_true, y_pred, f"medsentix_{variant}", project_root)
    save_training_curve(history, f"medsentix_{variant}", project_root)
    with checkpoint_path.with_suffix(".history.json").open("w", encoding="utf-8") as handle:
        json.dump(history, handle, indent=2)

    new_row = pd.DataFrame([metrics])
    result_df = new_row
    if save_variant_results:
        table_path = project_root / "results/tables/variant_comparison.csv"
        if table_path.exists():
            previous = pd.read_csv(table_path)
            previous = previous[previous["model"] != metrics["model"]]
            result_df = pd.concat([previous, new_row], ignore_index=True)
        result_df.to_csv(table_path, index=False)
    return model, result_df, history


def load_medsentix_checkpoint(
    variant: str,
    project_root: Path = PROJECT_ROOT,
    device: Optional[torch.device] = None,
    **model_kwargs,
) -> MedSentiX:
    """Load a saved MedSentiX checkpoint for evaluation or explainability."""
    device = device or get_device()
    model = MedSentiX(**model_kwargs).to(device)
    checkpoint_path = project_root / f"checkpoints/medsentix/medsentix_{variant}.pt"
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.eval()
    return model


def cross_dataset_generalization(
    model: MedSentiX,
    variant: str,
    project_root: Path,
    device: torch.device,
    dev_mode: bool = False,
    sample_size: int = 1000,
) -> float:
    """Average accuracy on held-out datasets not used directly by a variant."""
    heldout_by_variant = {
        "D": ["webmd", "druglib"],
        "DW": ["druglib"],
        "DDL": ["webmd"],
        "Full": [],
    }
    heldouts = heldout_by_variant.get(variant, [])
    if not heldouts:
        return math.nan
    tokenizer = AutoTokenizer.from_pretrained(BIOBERT_MODEL_NAME)
    accuracies: List[float] = []
    for dataset_name in heldouts:
        path = project_root / f"data/splits/{dataset_name}_test.csv"
        if not path.exists():
            continue
        frame = pd.read_csv(path)
        if dev_mode:
            frame = frame.head(sample_size)
        loader = DataLoader(MedSentiXDataset(standardize_frame(frame, dataset_name), tokenizer), batch_size=BATCH_SIZE)
        metrics, _, _, _ = evaluate_medsentix(model, loader, device)
        accuracies.append(float(metrics["accuracy"]))
    return float(np.mean(accuracies)) if accuracies else math.nan


def evaluate_all_medsentix_variants(
    project_root: Path = PROJECT_ROOT,
    dev_mode: bool = False,
    sample_size: int = 1000,
) -> pd.DataFrame:
    """Evaluate every saved MedSentiX checkpoint on its configured test split."""
    ensure_medsentix_dirs(project_root)
    device = get_device()
    rows = []
    for variant in ["D", "DW", "DDL", "Full"]:
        checkpoint_path = project_root / f"checkpoints/medsentix/medsentix_{variant}.pt"
        if not checkpoint_path.exists():
            print(f"Skipping MedSentiX-{variant}: missing checkpoint {checkpoint_path}")
            continue
        _, _, test_df = load_variant_splits(variant, project_root, dev_mode, sample_size)
        _, _, test_loader, _ = make_medsentix_loaders(test_df, test_df, test_df)
        model = load_medsentix_checkpoint(variant, project_root, device)
        metrics, y_true, y_pred, _ = evaluate_medsentix(model, test_loader, device)
        metrics["model"] = f"medsentix_{variant}"
        metrics["cross_dataset_generalization"] = cross_dataset_generalization(model, variant, project_root, device, dev_mode, sample_size)
        save_confusion_matrix(y_true, y_pred, f"medsentix_{variant}", project_root)
        rows.append(metrics)
    result_df = pd.DataFrame(rows)
    if not result_df.empty:
        result_df.to_csv(project_root / "results/tables/variant_comparison.csv", index=False)
    return result_df


def run_ablation_study(
    project_root: Path = PROJECT_ROOT,
    dev_mode: bool = True,
    sample_size: int = 1000,
    epochs: int = EPOCHS,
) -> pd.DataFrame:
    """Run focused ablations for the Full variant without changing headline settings."""
    rows = []
    configs = [
        {"ablation": "full_model", "use_bilstm": True, "use_attention": True, "use_auxiliary_heads": True, "lambda_aspect": LAMBDA_ASPECT},
        {"ablation": "no_auxiliary_guidance", "use_bilstm": True, "use_attention": True, "use_auxiliary_heads": True, "lambda_aspect": 0.0},
        {"ablation": "no_guided_attention", "use_bilstm": True, "use_attention": False, "use_auxiliary_heads": False, "lambda_aspect": 0.0},
        {"ablation": "no_bilstm", "use_bilstm": False, "use_attention": True, "use_auxiliary_heads": True, "lambda_aspect": LAMBDA_ASPECT},
    ]
    for config in configs:
        model, _, _ = train_medsentix_variant(
            "Full",
            project_root=project_root,
            dev_mode=dev_mode,
            sample_size=sample_size,
            epochs=epochs,
            lambda_aspect=config["lambda_aspect"],
            use_bilstm=config["use_bilstm"],
            use_attention=config["use_attention"],
            use_auxiliary_heads=config["use_auxiliary_heads"],
            checkpoint_name=f"ablation_{config['ablation']}",
            save_variant_results=False,
        )
        device = next(model.parameters()).device
        _, _, test_df = load_variant_splits("Full", project_root, dev_mode, sample_size)
        _, _, test_loader, _ = make_medsentix_loaders(test_df, test_df, test_df)
        metrics, _, _, _ = evaluate_medsentix(model, test_loader, device)
        rows.append({"ablation": config["ablation"], **metrics})
    result_df = pd.DataFrame(rows)
    result_df.to_csv(project_root / "results/tables/ablation_results.csv", index=False)
    return result_df


def validate_absa_on_druglib(
    variants: Sequence[str] = ("DDL", "Full"),
    project_root: Path = PROJECT_ROOT,
    dev_mode: bool = False,
    sample_size: int = 1000,
) -> pd.DataFrame:
    """Evaluate aspect heads on Druglib labels and save ABSA validation metrics."""
    ensure_medsentix_dirs(project_root)
    device = get_device()
    tokenizer = AutoTokenizer.from_pretrained(BIOBERT_MODEL_NAME)
    frame = pd.read_csv(project_root / "data/splits/druglib_test.csv")
    if dev_mode:
        frame = frame.head(sample_size)
    dataset = MedSentiXDataset(standardize_frame(frame, "druglib"), tokenizer)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE)
    rows = []
    for variant in variants:
        checkpoint_path = project_root / f"checkpoints/medsentix/medsentix_{variant}.pt"
        if not checkpoint_path.exists():
            print(f"Skipping ABSA validation for MedSentiX-{variant}: missing checkpoint.")
            continue
        model = load_medsentix_checkpoint(variant, project_root, device)
        aspect_true: Dict[str, List[int]] = {name: [] for name in ASPECT_NAMES}
        aspect_pred: Dict[str, List[int]] = {name: [] for name in ASPECT_NAMES}
        main_true: List[int] = []
        main_pred: List[int] = []
        model.eval()
        with torch.no_grad():
            for batch in loader:
                logits, _, auxiliary_logits = model(batch["input_ids"].to(device), batch["attention_mask"].to(device))
                main_true.extend(batch["labels"].tolist())
                main_pred.extend(torch.argmax(logits, dim=1).cpu().tolist())
                for index, (name, column) in enumerate(zip(ASPECT_NAMES, ASPECT_COLUMNS)):
                    labels = batch[column]
                    valid = labels.ne(MISSING_ASPECT_LABEL)
                    if valid.any() and index < len(auxiliary_logits):
                        aspect_true[name].extend(labels[valid].tolist())
                        aspect_pred[name].extend(torch.argmax(auxiliary_logits[index].detach().cpu()[valid], dim=1).tolist())
        row = {"model": f"medsentix_{variant}"}
        f1_values = []
        for name in ASPECT_NAMES:
            if aspect_true[name]:
                row[f"{name}_accuracy"] = accuracy_score(aspect_true[name], aspect_pred[name])
                row[f"{name}_f1"] = f1_score(aspect_true[name], aspect_pred[name], average="macro", zero_division=0)
                f1_values.append(row[f"{name}_f1"])
            elif name == "overall_satisfaction":
                row[f"{name}_accuracy"] = accuracy_score(main_true, main_pred)
                row[f"{name}_f1"] = f1_score(main_true, main_pred, average="macro", zero_division=0)
                f1_values.append(row[f"{name}_f1"])
            else:
                row[f"{name}_accuracy"] = math.nan
                row[f"{name}_f1"] = math.nan
        row["average_aspect_f1"] = float(np.nanmean(f1_values)) if f1_values else math.nan
        rows.append(row)
    result_df = pd.DataFrame(rows)
    result_df.to_csv(project_root / "results/tables/absa_validation.csv", index=False)
    return result_df


def attention_entropy(attention_weights: torch.Tensor, attention_mask: torch.Tensor) -> np.ndarray:
    """Compute normalized entropy for each attention head."""
    weights = attention_weights.detach().cpu()
    mask = attention_mask.detach().cpu().bool()
    entropies = []
    for head in range(weights.size(1)):
        head_weights = weights[:, head]
        valid_values = []
        for batch_index in range(head_weights.size(0)):
            valid_len = int(mask[batch_index].sum().item())
            if valid_len > 1:
                values = head_weights[batch_index, :valid_len, :valid_len].clamp(min=1e-12)
                entropy = -(values * values.log()).sum(dim=-1) / math.log(valid_len)
                valid_values.append(float(entropy.mean().item()))
        entropies.append(float(np.mean(valid_values)) if valid_values else math.nan)
    return np.asarray(entropies)


def save_attention_heatmap(
    model: MedSentiX,
    tokenizer,
    review: str,
    project_root: Path = PROJECT_ROOT,
    filename: str = "medsentix_full_attention_heatmap.png",
) -> None:
    """Save an aspect-head attention heatmap for one review."""
    ensure_medsentix_dirs(project_root)
    device = next(model.parameters()).device
    encoded = tokenizer(review, truncation=True, padding="max_length", max_length=MAX_LEN, return_tensors="pt")
    tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"].squeeze(0))[: int(encoded["attention_mask"].sum())]
    with torch.no_grad():
        _, attention, _ = model(encoded["input_ids"].to(device), encoded["attention_mask"].to(device))
    if attention is None:
        return
    attention = attention.detach().cpu()[0, :, : len(tokens), : len(tokens)].mean(dim=1).numpy()
    plt.figure(figsize=(max(8, len(tokens) * 0.25), 4))
    sns.heatmap(attention, cmap="viridis", xticklabels=tokens, yticklabels=ASPECT_NAMES)
    plt.xticks(rotation=90)
    plt.title("MedSentiX Aspect Attention")
    plt.tight_layout()
    plt.savefig(project_root / f"results/figures/attention_heatmaps/{filename}", dpi=300)
    plt.close()


def run_shap_analysis(
    project_root: Path = PROJECT_ROOT,
    dev_mode: bool = True,
    sample_size: int = 100,
) -> pd.DataFrame:
    """Run SHAP explainability for MedSentiX-Full and save paper figures."""
    ensure_medsentix_dirs(project_root)
    if shap is None:
        raise ImportError("shap is required for SHAP analysis.")
    device = get_device()
    model = load_medsentix_checkpoint("Full", project_root, device)
    tokenizer = AutoTokenizer.from_pretrained(BIOBERT_MODEL_NAME)
    frame = pd.read_csv(project_root / "data/splits/drugs_com_test.csv")
    if dev_mode:
        frame = frame.head(sample_size)
    texts = frame["review"].astype(str).tolist()

    def predict_proba(batch_texts: Sequence[str]) -> np.ndarray:
        encoded = tokenizer(
            list(batch_texts),
            truncation=True,
            padding=True,
            max_length=MAX_LEN,
            return_tensors="pt",
        )
        with torch.no_grad():
            logits, _, _ = model(encoded["input_ids"].to(device), encoded["attention_mask"].to(device))
        return torch.softmax(logits, dim=-1).detach().cpu().numpy()

    masker = shap.maskers.Text(tokenizer)
    explainer = shap.Explainer(predict_proba, masker, output_names=[ID2LABEL[i] for i in range(NUM_CLASSES)])
    shap_values = explainer(texts)

    shap.plots.bar(shap_values, max_display=20, show=False)
    plt.tight_layout()
    plt.savefig(project_root / "results/figures/shap_plots/global_top20_bar.png", dpi=300)
    plt.close()

    for class_index, class_name in ID2LABEL.items():
        shap.plots.bar(shap_values[:, :, class_index], max_display=20, show=False)
        plt.tight_layout()
        plt.savefig(project_root / f"results/figures/shap_plots/{class_name.lower()}_class_bar.png", dpi=300)
        plt.close()

    for index in range(min(3, len(texts))):
        shap.plots.waterfall(shap_values[index, :, int(frame.iloc[index]["label"])], max_display=20, show=False)
        plt.tight_layout()
        plt.savefig(project_root / f"results/figures/shap_plots/waterfall_review_{index + 1}.png", dpi=300)
        plt.close()

    entropy_rows = []
    for text in texts[: min(32, len(texts))]:
        encoded = tokenizer(text, truncation=True, padding="max_length", max_length=MAX_LEN, return_tensors="pt")
        with torch.no_grad():
            _, attention, _ = model(encoded["input_ids"].to(device), encoded["attention_mask"].to(device))
        if attention is not None:
            entropy_rows.append(attention_entropy(attention, encoded["attention_mask"]))
    entropy_df = pd.DataFrame(entropy_rows, columns=ASPECT_NAMES)
    entropy_summary = entropy_df.mean().reset_index()
    entropy_summary.columns = ["aspect_head", "attention_entropy"]
    entropy_summary.to_csv(project_root / "results/tables/shap_attention_entropy.csv", index=False)

    plt.figure(figsize=(7, 4))
    sns.barplot(data=entropy_summary, x="aspect_head", y="attention_entropy")
    plt.xticks(rotation=20, ha="right")
    plt.title("Attention Entropy by Aspect Head")
    plt.tight_layout()
    plt.savefig(project_root / "results/figures/shap_plots/attention_entropy_by_head.png", dpi=300)
    plt.close()
    return entropy_summary
'''
# The maintained MedSentiX implementation lives in models/medsentix.py.
from models.medsentix import (
    evaluate_all_medsentix_variants,
    run_ablation_study,
    run_shap_analysis,
    train_medsentix_variant,
    validate_absa_on_druglib,
)


## 5. MedSentiX-D

In [ ]:
model_D, results_D, history_D = train_medsentix_variant("D", PROJECT_ROOT, DEV_MODE, SAMPLE_SIZE, epochs=10)
display(results_D)
del model_D, results_D, history_D
cleanup_memory()


## 6. MedSentiX-DW

In [ ]:
model_DW, results_DW, history_DW = train_medsentix_variant("DW", PROJECT_ROOT, DEV_MODE, SAMPLE_SIZE, epochs=10)
display(results_DW)
del model_DW, results_DW, history_DW
cleanup_memory()


## 7. MedSentiX-DDL

In [ ]:
model_DDL, results_DDL, history_DDL = train_medsentix_variant("DDL", PROJECT_ROOT, DEV_MODE, SAMPLE_SIZE, epochs=10)
display(results_DDL)
del model_DDL, results_DDL, history_DDL
cleanup_memory()


## 8. MedSentiX-Full

In [ ]:
model_Full, results_Full, history_Full = train_medsentix_variant("Full", PROJECT_ROOT, DEV_MODE, SAMPLE_SIZE, epochs=10)
display(results_Full)
del model_Full, results_Full, history_Full
cleanup_memory()


## 9. Model Evaluation

In [ ]:
variant_results = evaluate_all_medsentix_variants(PROJECT_ROOT, dev_mode=DEV_MODE, sample_size=SAMPLE_SIZE)
all_model_results = pd.concat([baseline_results, variant_results], ignore_index=True, sort=False)
all_model_results.to_csv(PROJECT_ROOT / "results/tables/all_model_results.csv", index=False)
display(all_model_results)


## 10. Ablation Study

In [ ]:
ablation_results = run_ablation_study(PROJECT_ROOT, dev_mode=DEV_MODE, sample_size=SAMPLE_SIZE, epochs=10)
display(ablation_results)


## 11. ABSA Validation

In [ ]:
absa_results = validate_absa_on_druglib(("D", "DW", "DDL", "Full"), PROJECT_ROOT, DEV_MODE, SAMPLE_SIZE)
display(absa_results)


## 12. SHAP Explainability

In [ ]:
entropy_summary = run_shap_analysis(PROJECT_ROOT, dev_mode=DEV_MODE, sample_size=min(100, SAMPLE_SIZE))
display(entropy_summary)


## 13. Final Results
This section loads the calculated result tables and points to the generated figures. It does not invent or hardcode scores.

In [ ]:
for table_name in [
    "baseline_results.csv",
    "variant_comparison.csv",
    "all_model_results.csv",
    "ablation_results.csv",
    "absa_validation.csv",
    "shap_attention_entropy.csv",
]:
    path = PROJECT_ROOT / "results/tables" / table_name
    if path.exists():
        print("\n", table_name)
        display(pd.read_csv(path))
    else:
        print("Missing:", path)

figure_root = PROJECT_ROOT / "results/figures"
print("Generated figure files:")
for path in sorted(figure_root.rglob("*.png")):
    print(path.relative_to(PROJECT_ROOT))
